In [1]:
import pandas as pd

In [2]:
columns = ['ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
           'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
           'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
           'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
           'tunnel_parents', 'ip_proto']

In [3]:
df = pd.read_csv("../data/raw/zeek-captures/conn.log",sep='\t',comment='#', names = columns)

In [4]:
print(df.shape)
df.head()

(2540, 22)


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,1.790911e+09,C946Oi3wl6FPOzvrNb,192.168.150.129,41656,45.33.32.156,80,tcp,-,-,-,...,T,F,0,C,0,0,0,0,-,6
1,1.790911e+09,CLF8yz0b7W9O25B6e,192.168.150.129,41668,45.33.32.156,80,tcp,-,-,-,...,T,F,0,C,0,0,0,0,-,6
2,1.790911e+09,CJQ4HJ3oYwaRfQSIH6,192.168.150.129,40080,45.33.32.156,993,tcp,-,-,-,...,T,F,0,C,0,0,0,0,-,6
3,1.790911e+09,CIVhFN21dHJQcwI0H8,192.168.150.129,58012,45.33.32.156,110,tcp,-,-,-,...,T,F,0,C,0,0,0,0,-,6
4,1.790911e+09,ChMrInJuilfpmOjAb,192.168.150.129,45786,45.33.32.156,111,tcp,-,-,-,...,T,F,0,C,0,0,0,0,-,6


In [5]:
result = (
    df.groupby("id.orig_h")["id.resp_p"]
      .nunique()
      .reset_index(name="unique_ports")
)

In [6]:
result['is_scanner'] = result['unique_ports'] > 50

In [7]:
print(result)

         id.orig_h  unique_ports  is_scanner
0    192.168.150.1             1       False
1  192.168.150.129          1000        True
2     45.33.32.156           486        True


## Limitation: the detector also flags the scan *target*

The detector flags any host that contacted more than 50 distinct destination
ports. On this capture it correctly flags the scanner
(`192.168.150.129`, 1000 ports) — but it also flags `45.33.32.156`
(scanme.nmap.org, 486 ports), which was the **target** of the scan, not an
attacker.

**Why this happens:** the detector counts distinct destination ports per
source host, but a scan produces bidirectional traffic. The target responds
from many ports, so Zeek logs the target as a source associated with many
ports too. Distinct-port-count alone can't distinguish the host that
*initiated* the connections (the scanner) from the host that *responded*
(the victim).

**How it could be refined:** use connection direction to tell initiator from
responder — for example, the scanner is consistently the originator
(`id.orig_h`) of large numbers of failed or reset connections
(`conn_state` values like `RSTRH`/`REJ`/`S0`), while the target's traffic is
responses. Filtering to failed, outbound-initiated connections before counting
ports would isolate the true scanner.